In [ ]:
%pip install google-generativeai pandas openpyxl tqdm --quiet
print("Installation abgeschlossen.")

In [ ]:
import os, json, time, re
import pandas as pd
from tqdm import tqdm
import google.generativeai as genai


print("Imports erfolgreich.")

In [ ]:
GEMINI_API_KEY   = os.environ.get("GEMINI_API_KEY", "")
MISTRAL_API_KEY  = os.environ.get("MISTRAL_API_KEY", "")
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")

GEMINI_MODEL   = "gemini-2.5-flash-lite"
MISTRAL_MODEL  = "mistral-small-3.2"
DEEPSEEK_MODEL = "deepseek-chat"

GEMINI_DELAY   = 0.1
MISTRAL_DELAY   = 0.1
DEEPSEEK_DELAY = 0.1

INPUT_FILE          = "Filtered_Task_Statements.xlsx"
INPUT_SHEET         = 0
CHECKPOINT_GEMINI   = "checkpoint_gemini.csv"
CHECKPOINT_MISTRAL   = "checkpoint_mistral.csv"
CHECKPOINT_DEEPSEEK = "checkpoint_deepseek.csv"
OUTPUT_EXCEL        = "DDL_Binary_Step1.xlsx"

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel(GEMINI_MODEL)

mistral_client = MISTRAL_API_KEY(
    api_key=MISTRAL_API_KEY,
    base_url="https://api.mistral.ai/v1"
)

deepseek_client = DEEPSEEK_API_KEY(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

print(f"Gemini   : {GEMINI_MODEL}  (Delay: {GEMINI_DELAY}s)")
print(f"Mistral   : {MISTRAL_MODEL}  (Delay: {MISTRAL_DELAY}s)")
print(f"DeepSeek : {DEEPSEEK_MODEL}  (Delay: {DEEPSEEK_DELAY}s)")
print(f"Input    : {INPUT_FILE} / Sheet: {INPUT_SHEET}")
print(f"Output   : {OUTPUT_EXCEL}")

In [ ]:
TEST_OCC  = "Clinical Research Coordinators"
TEST_TASK = "Recruit and screen patients for Phase II clinical trials according to protocol inclusion/exclusion criteria."

def test_gemini():
    try:
        ddl_ctx = (
            "You classify occupational tasks. D=1 if the task has a DIRECT connection to the Drug Development Lifecycle "
            "(drug discovery, preclinical research, clinical trials Phase I-III, or FDA/EMA regulatory review). "
            "D=0 if generic (admin, IT, HR, etc.) even in pharma. Conservative: when in doubt → D=0. "
            'Respond ONLY with JSON: {"D": 0_or_1, "reason": "one sentence"}'
        )
        prompt = ddl_ctx + f"\n\nOccupation: {TEST_OCC}\nTask: \"{TEST_TASK}\""
        resp = gemini_model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(temperature=0, max_output_tokens=150)
        )
        raw = resp.text.strip()
        import re, json
        clean = re.sub(r'```(?:json)?\s*|```', '', raw).strip()
        parsed = json.loads(re.search(r'\{.*?\}', clean, re.DOTALL).group())
        print(f"  Gemini    : OK — D={parsed['D']}")
        print(f"              Reason: {str(parsed.get('reason',''))}")
        return True
    except Exception as e:
        print(f"  Gemini    : FEHLER — {e}")
        return False

def test_mistral():
    try:
        simple_sys = 'You classify occupational tasks. Respond ONLY with JSON: {"D": 1_or_0, "reason": "one sentence"}'
        simple_usr = f'Is this task related to drug development?\nOccupation: {TEST_OCC}\nTask: "{TEST_TASK}"'
        resp = mistral_client.chat.completions.create(
            model=MISTRAL_MODEL,
            messages=[
                {"role": "system", "content": simple_sys},
                {"role": "user",   "content": simple_usr}
            ],
            max_tokens=200, temperature=0
        )
        raw     = resp.choices[0].message.content or ""
        finish  = resp.choices[0].finish_reason
        if not raw.strip():
            print(f"  Mistral Small 3.2: WARNUNG — leere Antwort (finish={finish}).")
            return False
        import re, json
        clean = re.sub(r'```(?:json)?\s*|```', '', raw).strip()
        parsed = json.loads(re.search(r'\{.*?\}', clean, re.DOTALL).group())
        print(f"  Mistral Small 3.2: OK — D={parsed['D']}")
        print(f"              Reason: {str(parsed.get('reason',''))}")
        return True
    except Exception as e:
        print(f"  Mistral Small 3.2: FEHLER — {e}")
        return False

def test_deepseek():
    try:
        simple_sys = 'You classify occupational tasks. Respond ONLY with JSON: {"D": 1_or_0, "reason": "one sentence"}'
        simple_usr = f'Is this task related to drug development?\nOccupation: {TEST_OCC}\nTask: "{TEST_TASK}"'
        resp = deepseek_client.chat.completions.create(
            model=DEEPSEEK_MODEL,
            messages=[
                {"role": "system", "content": simple_sys},
                {"role": "user",   "content": simple_usr}
            ],
            max_tokens=200, temperature=0
        )
        raw = resp.choices[0].message.content
        import re, json
        clean = re.sub(r'```(?:json)?\s*|```', '', raw).strip()
        parsed = json.loads(re.search(r'\{.*?\}', clean, re.DOTALL).group())
        print(f"  DeepSeek  : OK — D={parsed['D']}")
        print(f"              Reason: {str(parsed.get('reason',''))}")
        return True
    except Exception as e:
        print(f"  DeepSeek  : FEHLER — {e}")
        return False

print("API-Tests (echter DDL-Task als Probe):")
print(f"  Occupation : {TEST_OCC}")
print(f"  Task       : {TEST_TASK[:70]}...")
print(f"  Erwartetes Ergebnis: D=1")
print()
ok_gemini   = test_gemini()
time.sleep(GEMINI_DELAY)
ok_mistral   = test_mistral()
ok_deepseek = test_deepseek()

print()
all_ok = ok_gemini and ok_mistral and ok_deepseek
if all_ok:
    print("✓ Alle APIs funktionieren. Weiter mit Schritt 4.")
else:
    print("✗ Mindestens eine API fehlerhaft. Bitte API Keys in Schritt 2 prüfen.")

In [ ]:
SYSTEM_PROMPT = """You are a scientific classifier for a bachelor's thesis on the impact of generative AI on the drug development lifecycle (DDL).

TASK: Classify whether an O*NET occupational task has a DIRECT AND SUBSTANTIAL connection to the Drug Development Lifecycle. The task must not be based on keywords solely but can also be interpreted semantically for its relationship to the DDL

DRUG DEVELOPMENT LIFECYCLE (DDL) — STRICT DEFINITION:
The DDL covers four stages:

1. DISCOVERY & DEVELOPMENT: Tasks and activitities related to Hypothesis-driven research to identify drug targets, design assays, screen molecular hits, optimize lead compounds. Includes: disease mechanism research, target validation, assay development, high-throughput screening, medicinal chemistry, early pharmacological profiling, preliminary safety assessments (pre-GLP).

2. PRECLINICAL RESEARCH: Tasks and activitities related to GLP-compliant laboratory studies (in vitro and in vivo/animal) evaluating safety, toxicity, and ADME (pharmacokinetics/pharmacodynamics) of drug candidates. Includes: formal toxicology studies, ADME characterization, IND-enabling studies, preparation of regulatory packages for first-in-human trials.

3. CLINICAL RESEARCH: Tasks and activitities related to Human trials (Phase I–III) evaluating safety, tolerability, dosing, and efficacy. Includes: clinical trial design, protocol development, patient recruitment, data collection, biostatistics, pharmacovigilance, clinical monitoring, regulatory submissions tied to trial conduct.

4. REGULATORY REVIEW (FDA/EMA): Tasks and activitities related to Submission and evaluation of NDA/MAA or equivalent. Includes: benefit-risk assessment, statistical analysis, labelling, manufacturing quality control (GMP), regulatory affairs for marketing authorization.

CLASSIFICATION RULES — STRICT:
D=1 (DDL-relevant) ONLY if the task has a DIRECT AND SUBSTANTIAL connection to DDL through:
  - Direct scientific/laboratory work on drug candidates or biological systems
  - Clinical trial conduct, design, or management
  - Regulatory submission or compliance for drug approval
  - Pharmacovigilance or drug safety monitoring
  - Manufacturing of investigational/pharmaceutical products (GMP context)
  - Data analysis specifically tied to drug development outcomes
  - Project/program management of DDL activities (not general project management)

D=0 (NOT DDL-relevant) if the task is:
  - Generic administrative work (scheduling, filing, correspondence) even in pharma companies
  - General IT/software development not specific to drug development systems
  - General HR, payroll, accounting, or financial tasks
  - General facility/building management
  - General marketing or sales (not drug regulatory/medical affairs)
  - Generic teaching/training not specific to DDL
  - Basic patient care not tied to clinical trials
  - Any task where the DDL connection is INDIRECT, tangential, or could apply equally to non-pharma contexts

CONSERVATIVE BIAS: When in doubt → D=0. Only clearly DDL-relevant tasks get D=1.

RESPOND ONLY with valid JSON, no additional text:
{"D": 0_or_1, "reason": "one concise sentence explaining the classification decision"}"""

FEW_SHOT_EXAMPLES = """
CALIBRATION EXAMPLES:

Occupation: Biochemists and Biophysicists
Task: "Design experiments to identify drug targets by analyzing protein-ligand interactions."
{"D": 1, "reason": "Direct Discovery & Development activity: drug target identification and assay design."}

Occupation: Regulatory Affairs Specialists
Task: "Compile and submit New Drug Applications (NDA) to the FDA, including clinical and preclinical data summaries."
{"D": 1, "reason": "Core Regulatory Review activity: NDA compilation and submission is the central DDL Stage 4 task."}

Occupation: Pharmaceutical Sales Representatives
Task: "Maintain customer records and update sales tracking databases for territory management."
{"D": 0, "reason": "Generic sales administration with no direct connection to drug development activities."}

Occupation: Medical Scientists
Task: "Write grant proposals to secure research funding from federal agencies."
{"D": 0, "reason": "Grant writing is a generic research administration task; not a DDL activity itself."}

Occupation: Clinical Pharmacologists
Task: "Analyze pharmacokinetic data from Phase I dose-escalation studies to determine maximum tolerated dose."
{"D": 1, "reason": "Direct Clinical Research activity: PK analysis in Phase I trials is a core DDL task."}
"""

def build_prompt(occupation: str, task: str) -> str:
    return (
        f"{FEW_SHOT_EXAMPLES}\n"
        f"NOW CLASSIFY:\n"
        f"Occupation: {occupation}\n"
        f'Task: "{task}"\n'
    )

print(f"System-Prompt : {len(SYSTEM_PROMPT):,} Zeichen")
print(f"Few-Shot-Block : {len(FEW_SHOT_EXAMPLES):,} Zeichen")
print(f"Gesamt/Task   : ~{len(SYSTEM_PROMPT) + len(FEW_SHOT_EXAMPLES) + 100:,} Zeichen (Input)")
print("Prompt bereit.")

In [ ]:
df = pd.read_excel(INPUT_FILE, sheet_name=INPUT_SHEET)
df["Task ID"] = df["Task ID"].astype(str)

print(f"Geladen: {len(df):,} Tasks aus {df['Title'].nunique():,} Occupations")
print(f"Spalten: {list(df.columns)}")
print(f"Task Types: {df['Task Type'].value_counts().to_dict()}")
print()
print("Erste 3 Tasks zur Kontrolle:")
print(df[["Task ID", "Title", "Task", "Task Type"]].head(3).to_string(index=False))

In [ ]:
def parse_json_response(text: str) -> dict:
    """Parst JSON aus Model-Antwort, auch wenn Markdown-Fences vorhanden."""
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'```\s*$', '', text, flags=re.MULTILINE)
    text = text.strip()
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError(f"Kein JSON gefunden in: {text[:100]}")

def query_gemini(occupation: str, task: str) -> dict:
    """Sendet Task an Gemini 2.5 Flash-Lite. Gibt {D: 0/1/-1, reason: str} zurück."""
    try:
        prompt = SYSTEM_PROMPT + "\n\n" + build_prompt(occupation, task)
        resp = gemini_model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(temperature=0, max_output_tokens=120)
        )
        result = parse_json_response(resp.text)
        return {"D": int(result["D"]), "reason": str(result.get("reason", ""))[:300]}
    except Exception as e:
        return {"D": -1, "reason": f"ERROR: {str(e)[:100]}"}

def query_mistral(occupation: str, task: str) -> dict:
    """Sendet Task an Mistral Small 3.2. Gibt {D: 0/1/-1, reason: str} zurück."""
    try:
        resp = mistral_client.chat.completions.create(
            model=MISTRAL_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_prompt(occupation, task)}
            ],
            max_tokens=120, temperature=0
        )
        result = parse_json_response(resp.choices[0].message.content)
        return {"D": int(result["D"]), "reason": str(result.get("reason", ""))[:300]}
    except Exception as e:
        return {"D": -1, "reason": f"ERROR: {str(e)[:100]}"}

def query_deepseek(occupation: str, task: str) -> dict:
    """Sendet Task an DeepSeek V3.2. Gibt {D: 0/1/-1, reason: str} zurück."""
    try:
        resp = deepseek_client.chat.completions.create(
            model=DEEPSEEK_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_prompt(occupation, task)}
            ],
            max_tokens=120, temperature=0
        )
        result = parse_json_response(resp.choices[0].message.content)
        return {"D": int(result["D"]), "reason": str(result.get("reason", ""))[:300]}
    except Exception as e:
        return {"D": -1, "reason": f"ERROR: {str(e)[:100]}"}

print("Klassifikations-Funktionen bereit.")
print()
print("Kurzer Funktionstest (1 Beispiel-Task):")
test_occ  = "Biochemists and Biophysicists"
test_task = "Design experiments to identify drug targets by analyzing protein-ligand interactions."

g = query_gemini(test_occ, test_task)
print(f"  Gemini   : D={g['D']}  — {g['reason'][:80]}")
time.sleep(GEMINI_DELAY)

o = query_mistral test_occ, test_task)
print(f"  Mistral Small 3.2: D={o['D']}  — {o['reason'][:80]}")

d = query_deepseek(test_occ, test_task)
print(f"  DeepSeek : D={d['D']}  — {d['reason'][:80]}")

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

if not (ok_gemini and ok_mistral and ok_deepseek):
    print("STOP: Nicht alle APIs funktionieren. Bitte Schritt 3 prüfen.")
else:
    def load_checkpoint(path):
        if os.path.exists(path):
            ck = pd.read_csv(path, dtype={"Task ID": str})
            done_ids = set(ck["Task ID"].tolist())
            print(f"  Resume aus {path}: {len(done_ids):,} Tasks bereits fertig.")
            return ck, done_ids
        return pd.DataFrame(), set()

    print("Lade Checkpoints...")
    df_g, done_g = load_checkpoint(CHECKPOINT_GEMINI)
    df_o, done_o = load_checkpoint(CHECKPOINT_MISTRAL)
    df_d, done_d = load_checkpoint(CHECKPOINT_DEEPSEEK)

    all_done = done_g & done_o & done_d
    remaining = df[~df["Task ID"].isin(all_done)].copy().reset_index(drop=True)

    secs_per_task = max(GEMINI_DELAY + 0.8, 1.0)
    est_total_sec = len(remaining) * secs_per_task
    est_h = int(est_total_sec // 3600)
    est_m = int((est_total_sec % 3600) // 60)

    print(f"\nGesamt Tasks      : {len(df):,}")
    print(f"Vollständig fertig: {len(all_done):,}")
    print(f"Noch ausstehend   : {len(remaining):,}")
    print(f"Geschätzte Zeit   : ~{est_h}h {est_m}min  (parallel, Delay={GEMINI_DELAY}s)")

    if len(remaining) == 0:
        print("\n✓ Alle Tasks bereits klassifiziert! Weiter mit Schritt 8.")
    else:
        buf_g, buf_o, buf_d = [], [], []
        errors_g = errors_o = errors_d = 0
        lock = threading.Lock()

        def classify_task(row):
            """Ruft alle drei Modelle PARALLEL auf für einen Task."""
            tid = str(row["Task ID"])
            occ = str(row["Title"])
            tsk = str(row["Task"])
            results = {}

            with ThreadPoolExecutor(max_workers=3) as ex:
                futures = {}
                if tid not in done_g:
                    futures["gemini"]   = ex.submit(query_gemini,   occ, tsk)
                if tid not in done_o:
                    futures["mistral"]   = ex.submit(query_mistral,   occ, tsk)
                if tid not in done_d:
                    futures["deepseek"] = ex.submit(query_deepseek, occ, tsk)
                for name, fut in futures.items():
                    results[name] = fut.result()

            return tid, results

        start_time = time.time()

        for i, (_, row) in enumerate(tqdm(remaining.iterrows(),
                                          total=len(remaining),
                                          desc="Klassifikation (parallel)")):
            tid, results = classify_task(row)

            if "gemini" in results:
                r = results["gemini"]
                if r["D"] == -1: errors_g += 1
                buf_g.append({"Task ID": tid, "D_gemini": r["D"], "reason_gemini": r["reason"]})

            if "mistral" in results:
                r = results["mistral"]
                if r["D"] == -1: errors_o += 1
                buf_o.append({"Task ID": tid, "D_mistral": r["D"], "reason_mistral": r["reason"]})

            if "deepseek" in results:
                r = results["deepseek"]
                if r["D"] == -1: errors_d += 1
                buf_d.append({"Task ID": tid, "D_deepseek": r["D"], "reason_deepseek": r["reason"]})

            time.sleep(GEMINI_DELAY)

            n_done = len(all_done) + i + 1
            if (i + 1) % 100 == 0:
                elapsed = time.time() - start_time
                rate    = (i + 1) / elapsed
                eta_sec = (len(remaining) - i - 1) / rate
                eta_h   = int(eta_sec // 3600)
                eta_m   = int((eta_sec % 3600) // 60)

                if buf_g:
                    pd.concat([df_g, pd.DataFrame(buf_g)], ignore_index=True)\
                      .drop_duplicates(subset="Task ID", keep="last")\
                      .to_csv(CHECKPOINT_GEMINI, index=False)
                if buf_o:
                    pd.concat([df_o, pd.DataFrame(buf_o)], ignore_index=True)\
                      .drop_duplicates(subset="Task ID", keep="last")\
                      .to_csv(CHECKPOINT_MISTRAL, index=False)
                if buf_d:
                    pd.concat([df_d, pd.DataFrame(buf_d)], ignore_index=True)\
                      .drop_duplicates(subset="Task ID", keep="last")\
                      .to_csv(CHECKPOINT_DEEPSEEK, index=False)

                tqdm.write(f"  ✓ {n_done:,} Tasks | ETA: ~{eta_h}h {eta_m}min | "
                           f"Fehler G={errors_g} O={errors_o} D={errors_d}")

        if buf_g:
            pd.concat([df_g, pd.DataFrame(buf_g)], ignore_index=True)\
              .drop_duplicates(subset="Task ID", keep="last")\
              .to_csv(CHECKPOINT_GEMINI, index=False)
        if buf_o:
            pd.concat([df_o, pd.DataFrame(buf_o)], ignore_index=True)\
              .drop_duplicates(subset="Task ID", keep="last")\
              .to_csv(CHECKPOINT_MISTRAL, index=False)
        if buf_d:
            pd.concat([df_d, pd.DataFrame(buf_d)], ignore_index=True)\
              .drop_duplicates(subset="Task ID", keep="last")\
              .to_csv(CHECKPOINT_DEEPSEEK, index=False)

        total_elapsed = time.time() - start_time
        print(f"\nKlassifikation abgeschlossen in {total_elapsed/60:.1f} Minuten.")
        print(f"  Fehler Gemini   : {errors_g:,}")
        print(f"  Fehler Mistral Small 3.2: {errors_o:,}")
        print(f"  Fehler DeepSeek : {errors_d:,}")
        print("Weiter mit Schritt 8.")

In [ ]:
missing = []
for path in [CHECKPOINT_GEMINI, CHECKPOINT_MISTRAL, CHECKPOINT_DEEPSEEK]:
    if not os.path.exists(path):
        missing.append(path)

if missing:
    print(f"Folgende Checkpoint-Dateien fehlen: {missing}")
    print("Bitte zuerst Schritt 7 ausführen.")
else:
    ck_g = pd.read_csv(CHECKPOINT_GEMINI,   dtype={"Task ID": str})
    ck_o = pd.read_csv(CHECKPOINT_MISTRAL,   dtype={"Task ID": str})
    ck_d = pd.read_csv(CHECKPOINT_DEEPSEEK, dtype={"Task ID": str})

    print(f"Gemini   Ergebnisse: {len(ck_g):,} Tasks")
    print(f"Mistral Small 3.2 Ergebnisse: {len(ck_o):,} Tasks")
    print(f"DeepSeek Ergebnisse: {len(ck_d):,} Tasks")
    print()

    merged = df.copy()
    merged = merged.merge(ck_g[["Task ID", "D_gemini", "reason_gemini"]],     on="Task ID", how="left")
    merged = merged.merge(ck_o[["Task ID", "D_mistral", "reason_mistral"]], on="Task ID", how="left")
    merged = merged.merge(ck_d[["Task ID", "D_deepseek", "reason_deepseek"]], on="Task ID", how="left")

    for col in ["D_gemini", "D_mistral", "D_deepseek"]:
        merged[col] = merged[col].fillna(-1).astype(int)

    def safe_vote(v): return 1 if v == 1 else 0

    merged["vote_sum"] = (
        merged["D_gemini"].apply(safe_vote) +
        merged["D_mistral"].apply(safe_vote) +
        merged["D_deepseek"].apply(safe_vote)
    )
    merged["D_final"] = (merged["vote_sum"] >= 2).astype(int)
    merged["n_models_agree"] = merged.apply(
        lambda r: sum(1 for v in [r["D_gemini"], r["D_mistral"], r["D_deepseek"]]
                      if safe_vote(v) == r["D_final"]),
        axis=1
    )

    total   = len(merged)
    ddl_rel = merged["D_final"].sum()
    unanimous_1 = (merged["vote_sum"] == 3).sum()
    unanimous_0 = (merged["vote_sum"] == 0).sum()
    split_votes = ((merged["vote_sum"] == 1) | (merged["vote_sum"] == 2)).sum()
    errors_any  = ((merged["D_gemini"] == -1) | (merged["D_mistral"] == -1) | (merged["D_deepseek"] == -1)).sum()

    print("=" * 55)
    print(f"  Gesamt Tasks               : {total:,}")
    print(f"  DDL-relevant (D_final=1)   : {ddl_rel:,}  ({100*ddl_rel/total:.1f}%)")
    print(f"  Nicht relevant (D_final=0) : {total-ddl_rel:,}  ({100*(total-ddl_rel)/total:.1f}%)")
    print(f"  Einstimmig D=1 (3/3)       : {unanimous_1:,}")
    print(f"  Einstimmig D=0 (0/3)       : {unanimous_0:,}")
    print(f"  Split-Vote (1/3 oder 2/3)  : {split_votes:,}")
    print(f"  Tasks mit Fehlern (D=-1)   : {errors_any:,}")
    print("=" * 55)

    print()
    print("Einzelmodell-Verteilung:")
    for col, label in [("D_gemini", "Gemini   "), ("D_mistral", "Mistral Small 3.2"), ("D_deepseek", "DeepSeek ")]:
        n1     = (merged[col] == 1).sum()
        n0     = (merged[col] == 0).sum()
        n_err  = (merged[col] == -1).sum()
        print(f"  {label}: D=1 → {n1:,}  |  D=0 → {n0:,}  |  Fehler → {n_err:,}")

    print()
    print("Inter-Rater Agreement (Paarweise Übereinstimmung):")
    valid = merged[(merged["D_gemini"] != -1) & (merged["D_mistral"] != -1) & (merged["D_deepseek"] != -1)]
    ag_go = (valid["D_gemini"]   == valid["D_mistral"]).mean()
    ag_gd = (valid["D_gemini"]   == valid["D_deepseek"]).mean()
    ag_od = (valid["D_mistral"] == valid["D_deepseek"]).mean()
    print(f"  Gemini ↔ Mistral Small 3.2 : {ag_go:.1%}")
    print(f"  Gemini ↔ DeepSeek   : {ag_gd:.1%}")
    print(f"  Mistral Small 3.2 ↔ DeepSeek: {ag_od:.1%}")
    print(f"  Alle drei einig     : {(valid['vote_sum'].isin([0,3])).mean():.1%}")

In [ ]:
if "merged" not in dir():
    print("Zuerst Schritt 8 ausführen.")
else:
    out = merged.sort_values(["Title", "Task ID"]).reset_index(drop=True)
    ddl_out = out[out["D_final"] == 1].reset_index(drop=True)

    base_cols = [c for c in ["O*NET-SOC Code", "Title", "Task ID", "Task", "Task Type",
                               "Incumbents Responding", "Date", "Domain Source"] if c in out.columns]
    result_cols = ["D_final", "vote_sum", "n_models_agree",
                   "D_gemini", "D_mistral", "D_deepseek",
                   "reason_gemini", "reason_mistral", "reason_deepseek"]
    result_cols = [c for c in result_cols if c in out.columns]
    export_cols = base_cols + result_cols

    occ_group_cols = [c for c in ["O*NET-SOC Code", "Title"] if c in out.columns]
    occ_sum = (
        out.groupby(occ_group_cols)
        .agg(
            total_tasks   = ("Task ID",    "count"),
            ddl_tasks     = ("D_final",    "sum"),
            ddl_pct       = ("D_final",    lambda x: round(x.mean() * 100, 1)),
            unanimous_D1  = ("vote_sum",   lambda x: (x == 3).sum()),
            split_votes   = ("vote_sum",   lambda x: x.isin([1, 2]).sum()),
        )
        .reset_index()
        .sort_values("ddl_pct", ascending=False)
    )

    with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
        ddl_out[export_cols].to_excel(writer, sheet_name="DDL_relevant",       index=False)
        out[export_cols].to_excel(    writer, sheet_name="All_Tasks",           index=False)
        occ_sum.to_excel(             writer, sheet_name="Occupation_Summary",  index=False)

    print(f"Exportiert: {OUTPUT_EXCEL}")
    print(f"  DDL_relevant      : {len(ddl_out):,} Tasks")
    print(f"  All_Tasks         : {len(out):,} Tasks")
    print(f"  Occupation_Summary: {len(occ_sum):,} Occupations")

In [ ]:
total_tasks = len(df) if "df" in dir() else "?"

print("=" * 55)
print("  FORTSCHRITT ÜBERSICHT")
print("=" * 55)

for path, label, col in [
    (CHECKPOINT_GEMINI,   "Gemini   ", "D_gemini"),
    (CHECKPOINT_MISTRAL,   "Mistral Small 3.2", "D_mistral"),
    (CHECKPOINT_DEEPSEEK, "DeepSeek ", "D_deepseek"),
]:
    if os.path.exists(path):
        ck = pd.read_csv(path, dtype={"Task ID": str})
        done = len(ck)
        errors = (ck[col] == -1).sum()
        d1 = (ck[col] == 1).sum()
        pct = f"{100*done/total_tasks:.1f}%" if isinstance(total_tasks, int) else "?"
        print(f"  {label}: {done:,} / {total_tasks}  ({pct})  |  D=1: {d1:,}  |  Fehler: {int(errors):,}")
    else:
        print(f"  {label}: kein Checkpoint")

print("=" * 55)